## Auto Commenter tool

The aim of this project is to create a code tool that automatically adds docstring/comments.

In [ ]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI(api_key=google_api_key, base_url=gemini_url)


MODEL = "gemini-3.5-flash"

In [ ]:
system_prompt = """
You are an expert in writing and understanding code. Your task is to go
through code and add clear, concise comments and docstrings explaining the reasoning,
not just restating what each line does. Respond only with the code and the added comments. Do not add
extra prose before or after.
"""

def user_prompt_for(code):
   return f"""Add brief comments and docstrings to the following code so that anybody who
reads it understands the thought process. Respond only with the code and comments, no
explanation outside the code block.
{code}
"""

In [ ]:
def messages_for(code):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(code)}
    ]
 

In [ ]:
def add_comments(code):
    """Stream commented code back from the model, yielding partial output as it arrives."""
    if not code or not code.strip():
        yield "Please paste some code first."
        return

    try:
        stream = openai.chat.completions.create(
            model=MODEL,
            messages=messages_for(code),
            stream=True
        )
        result = ""
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ""
            result += delta
            yield result
    except Exception as e:
        yield f"Error calling the model: {e}"

In [ ]:
with gr.Blocks(title="Auto Commenter") as demo:
    gr.Markdown("## Auto Commenter : paste code, get it back with comments/docstrings")
    with gr.Row():
        code_input = gr.Textbox(label="Your code", lines=20, placeholder="Paste code here...")
        code_output = gr.Textbox(label="Commented code", lines=20)
    submit_btn = gr.Button("Add Comments", variant="primary")
    submit_btn.click(fn=add_comments, inputs=code_input, outputs=code_output)

if __name__ == "__main__":
    demo.launch(share = True)